In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('IMDB Dataset.csv')

In [3]:
df.shape

(50000, 2)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()
df.drop_duplicates(inplace=True)

In [6]:
df.shape

(49582, 2)

In [7]:
#preprocessing the data
df['review'] = df['review'].str.lower()



### REMOVE THE URLS


In [8]:
import re
sample="abc is the word,abc"
new_text = re.sub(r'abc', 'xyz', sample)

In [9]:
def remove_urls(text):
    return re.sub(r'http\S+', '', text)

df['review'] = df['review'].apply(remove_urls)

In [10]:
def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)
df['review'] = df['review'].apply(remove_punctuation)

In [11]:
def remove_HTML_tags(text):
    return re.sub(r'<.*?>', '', text)
df['review'] = df['review'].apply(remove_HTML_tags)

In [12]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab') #sentence tokenization
nltk.download('wordnet') #lemmatization


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aayus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aayus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\aayus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\aayus\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [13]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

In [14]:
sample_text = "This is a sample sentence, showing off the stop words filtration."
word_tokens = word_tokenize(sample_text)    


In [15]:
def remove_stopwords(text):
   token=word_tokenize(text)
   stop_words = set(stopwords.words('english'))

   for word in token:
       if word in stop_words:
           text=text.replace(word, '')
   return text

### Stemming is the process of reducing a word to its base or root form. The most common stemming algorithm is the Porter Stemmer, which removes common suffixes from words. For example, "running" would be reduced to "run", and "happiness".

In [16]:
from nltk.stem import PorterStemmer

In [17]:
def stemming(text):
    ps = PorterStemmer()
    Stemmed_words = []
    token=word_tokenize(text)
    for word in token:
        Stemmed_words.append(ps.stem(word))
    return ' '.join(Stemmed_words)

df['review'] = df['review'].apply(stemming)
    

In [18]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])


In [19]:
y=df['sentiment']

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features=5000)
x=tf.fit_transform(df['review'])

### Dataset and Data Loaders

In [26]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x,
                                                    y, 
                                                    test_size=0.2, 
                                                    random_state=42)


In [27]:
import torch
from torch.utils.data import TensorDataset, DataLoader


In [28]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [29]:
train_set = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train.values, dtype=torch.long))
test_set = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test.values, dtype=torch.long))

In [30]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)


### BUILD OUR RNN MODEL

In [31]:
import torch.nn as nn
import torch.optim as optimizers

In [32]:
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
    
        self.RNNModel = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.RNNModel(x, h0)
        out = self.fc(out[:, -1, :])
        return out
        

In [33]:
input_size = X_train.shape[1]
model = RNNModel(input_size=input_size)
criterion = nn.CrossEntropyLoss()
optimizer = optimizers.Adam(model.parameters(), lr=0.001)

### Training the RNN Model

In [35]:
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.unsqueeze(1)  # Add sequence dimension
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/5], Loss: 0.3439
Epoch [2/5], Loss: 0.2914
Epoch [3/5], Loss: 0.2109
Epoch [4/5], Loss: 0.1807
Epoch [5/5], Loss: 0.2391


In [36]:
import json
import joblib

model.eval()

with torch.no_grad():
    correct = 0
    total = 0
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.unsqueeze(1)  # Add sequence dimension
        outputs = model(X_batch)
        predicted = torch.argmax(outputs, dim=1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

    accuracy = correct / total
    print(f'Test Accuracy: {accuracy:.4f}')

torch.save(model.state_dict(), 'rnn_model_state.pt')
joblib.dump(tf, 'tfidf_vectorizer.pkl')
joblib.dump(le, 'label_encoder.pkl')

with open('rnn_model_config.json', 'w', encoding='utf-8') as f:
    json.dump({'input_size': int(input_size), 'hidden_size': 128, 'num_layers': 1}, f)

print('Saved artifacts: rnn_model_state.pt, tfidf_vectorizer.pkl, label_encoder.pkl, rnn_model_config.json')

Test Accuracy: 0.8797
Saved artifacts: rnn_model_state.pt, tfidf_vectorizer.pkl, label_encoder.pkl, rnn_model_config.json
